# Baselines and Hyperparameter Selection



This notebook performs the hyperparameter search described in Section 3.5 and
establishes the untruncated reference performance of all three models. It runs
before the initialization and truncation studies because both of those depend on
its output: the configurations selected here are frozen and reused unchanged for
every subsequent condition.

The search follows the categories set out in Section 3.5.4. The truncation point
and the initialization method are independent variables and are *not* searched —
they are held at $k = T$ and Gaussian respectively, so that the nuisance
parameters are selected under a single fixed condition and can then be frozen.
Re-tuning them per condition would fold tuning effects into the very curves the
study is trying to read.

Two decisions from Section 3.5 shape what follows:

- **Substrate parameters are shared, optimisation parameters are not.** The leak
  rate and firing threshold define the substrate, so the LSM and the proposed
  model must adopt the same values or they would differ in two respects and the
  comparison between them would no longer isolate reservoir training. They are
  therefore searched on the LSM and inherited by the SNN — the conservative
  direction. The learning rate belongs to the optimisation problem rather than
  the substrate and is tuned per model.
- **The search is exhaustive.** A staged coordinate-descent alternative was
  measured on ECG5000 and missed the optimum by 0.0102 Macro-F1, its selection
  ranking tenth of ninety-six, so the separability it assumes does not hold here.

In [1]:
import pandas as pd

from edgesnn.data_utils import make_datasets
from edgesnn.experiments import (
    RESULTS_DIR, build_model, ensure_results_dir, load_config, run_grid, save_config,
)
from edgesnn.training import SELECTION_FOLDS
from edgesnn.tuning import BATCH_SIZES, DEFAULT_GRIDS, expand_grid, grid_search

In [2]:
DATA_DIR = "../Data"
ensure_results_dir()
SEED = 0

# InsectSound is by far the most expensive: one exhaustive SNN search there is on
# the order of half a day even on the GPU backend. Start with the two cheap
# datasets, confirm the pipeline, then add InsectSound to this list.
DATASETS = ["ECG5000", "Wafer", "InsectSound"]

for kind, grid in DEFAULT_GRIDS.items():
    print(f"{kind:<5} {len(expand_grid(grid)):>3} configurations")
print("\nselection folds:", SELECTION_FOLDS)
print("batch sizes:    ", BATCH_SIZES)

snn    96 configurations
lsm    24 configurations
lstm   10 configurations

selection folds: {'ECG5000': 5, 'Wafer': 5, 'InsectSound': 1}
batch sizes:     {'ECG5000': 64, 'Wafer': 64, 'InsectSound': 256}


In [3]:
splits = {}
for name in DATASETS:
    train_ds, test_ds = make_datasets(DATA_DIR, name)
    splits[name] = (train_ds, test_ds)
    print(f"{name:<12} train={len(train_ds):>6}  test={len(test_ds):>6}  "
          f"T={train_ds.seq_len:>4}  C={train_ds.num_classes}")

ECG5000      train=   500  test=  4500  T= 140  C=2
Wafer        train=  1000  test=  6164  T= 152  C=2
InsectSound  train= 25000  test= 25000  T= 600  C=10


## 1. Substrate search on the LSM

The neuron parameters are selected first, on the Liquid State Machine. Because
its reservoir is frozen, no gradient passes through a spike, and the search is
correspondingly cheap; more importantly, selecting them here and handing them to
the proposed model means no part of any later advantage can be attributed to a
substrate tuned in the proposed model's favour.

Selection is by Macro-F1 on a validation partition drawn from the training split,
cross-validated over five folds on ECG5000 and Wafer. The standard deviation
across folds is reported alongside the mean, because a difference between two
configurations is only meaningful when it exceeds the variation between folds.

In [4]:
substrate = {}

for name, (train_ds, _) in splits.items():
    cached = load_config(f"config_lsm_{name}")
    if cached is not None:
        substrate[name] = cached
        print(f"{name}: loaded cached configuration {cached}")
        continue

    table = grid_search(
        lambda **kw: build_model("lsm", num_classes=train_ds.num_classes,
                                 k=train_ds.seq_len, seed=SEED, **kw),
        train_ds, DEFAULT_GRIDS["lsm"], seed=SEED, desc=f"LSM · {name}",
        train_kwargs={"batch_size": BATCH_SIZES[name]},
    )
    table.to_csv(f"{RESULTS_DIR}/search_lsm_{name}.csv", index=False)

    best = table.iloc[0]
    substrate[name] = {"leak_rate": float(best.leak_rate),
                       "threshold": float(best.threshold),
                       "lr": float(best.lr)}
    save_config(f"config_lsm_{name}", substrate[name])
    display(table.head(5))

LSM · ECG5000:   0%|          | 0/24 [00:00<?, ?cfg/s]

  cfg 1/24 fold 1/5:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 1/24 fold 2/5:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 1/24 fold 3/5:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 1/24 fold 4/5:   0%|          | 0/200 [00:00<?, ?ep/s]

...

  cfg 21/24:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 22/24:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 23/24:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 24/24:   0%|          | 0/200 [00:00<?, ?ep/s]

,lr,leak_rate,threshold,val_macro_f1,val_macro_f1_std,mean_best_epoch,stopped_early,n_splits
0,0.0003,0.90,1.0,0.392535,0.0,58,True,1
1,0.0001,0.90,1.0,0.389904,0.0,82,True,1
2,0.0001,0.80,1.0,0.373133,0.0,69,True,1
3,0.0010,0.90,1.0,0.367895,0.0,38,True,1
4,0.0003,0.95,1.0,0.367110,0.0,69,True,1


## 2. Learning rate and surrogate for the proposed model


The substrate settled, the proposed model is searched over its own learning rate
and surrogate family, with the leak rate and threshold pinned to the values
inherited from the LSM.

The two are searched jointly rather than in sequence. As Section 2.3.2 derives,
the four surrogate families return derivatives of $1$, $\gamma/2$, $1$ and
$\gamma/4$ at the threshold, so at $\gamma = 10$ their gradient magnitudes differ
by a factor of five; a family evaluated at a learning rate suited to another
would be judged on its gradient scale rather than on its shape.

In [5]:
snn_config = {}

for name, (train_ds, _) in splits.items():
    cached = load_config(f"config_snn_{name}")
    if cached is not None:
        snn_config[name] = cached
        print(f"{name}: loaded cached configuration {cached}")
        continue

    fixed = {k: substrate[name][k] for k in ("leak_rate", "threshold")}
    grid = {"lr": DEFAULT_GRIDS["snn"]["lr"],
            "surrogate": DEFAULT_GRIDS["snn"]["surrogate"]}

    table = grid_search(
        lambda **kw: build_model("snn", num_classes=train_ds.num_classes,
                                 k=train_ds.seq_len, seed=SEED, **fixed, **kw),
        train_ds, grid, seed=SEED, desc=f"SNN · {name}", train_kwargs={"batch_size": BATCH_SIZES[name]},
    )
    table.to_csv(f"{RESULTS_DIR}/search_snn_{name}.csv", index=False)

    best = table.iloc[0]
    snn_config[name] = {**fixed, "surrogate": str(best.surrogate),
                        "lr": float(best.lr)}
    save_config(f"config_snn_{name}", snn_config[name])
    display(table.head(5))

SNN · ECG5000:   0%|          | 0/16 [00:00<?, ?cfg/s]

  cfg 1/16 fold 1/5:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 1/16 fold 2/5:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 1/16 fold 3/5:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 1/16 fold 4/5:   0%|          | 0/200 [00:00<?, ?ep/s]

...

  cfg 13/16:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 14/16:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 15/16:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 16/16:   0%|          | 0/200 [00:00<?, ?ep/s]

,lr,surrogate,val_macro_f1,val_macro_f1_std,mean_best_epoch,stopped_early,n_splits
0,0.0010,sigmoid,0.458983,0.0,37,True,1
1,0.0001,triangular,0.449724,0.0,125,True,1
2,0.0003,sigmoid,0.445897,0.0,51,True,1
3,0.0003,fast_sigmoid,0.442724,0.0,50,True,1
4,0.0010,triangular,0.440451,0.0,47,True,1


The interaction between learning rate and surrogate family can be read directly
off the search table. If each family attains its best score at a different
learning rate, the two could not have been tuned independently without
misjudging at least one of them.

In [6]:
for name in splits:
    table = pd.read_csv(f"{RESULTS_DIR}/search_snn_{name}.csv")
    pivot = table.pivot_table(index="surrogate", columns="lr", values="val_macro_f1")
    print(f"\n{name}: validation Macro-F1 by surrogate family and learning rate")
    display(pivot.round(4))
    print("best learning rate per family:")
    display(table.loc[table.groupby("surrogate").val_macro_f1.idxmax()]
            [["surrogate", "lr", "val_macro_f1", "val_macro_f1_std"]]
            .reset_index(drop=True))


ECG5000: validation Macro-F1 by surrogate family and learning rate


lr,0.0001,0.0003,0.0010,0.0030
surrogate,,,,
arctan,0.9711,0.9753,0.9835,0.9815
fast_sigmoid,0.9752,0.9773,0.9794,0.9835
sigmoid,0.9815,0.9815,0.9836,0.9835
triangular,0.9773,0.9835,0.9835,0.9855


best learning rate per family:


,surrogate,lr,val_macro_f1,val_macro_f1_std
0,arctan,0.001,0.983493,0.014010
1,fast_sigmoid,0.003,0.983503,0.012357
2,sigmoid,0.001,0.983597,0.012269
3,triangular,0.003,0.985549,0.010565



Wafer: validation Macro-F1 by surrogate family and learning rate


...

,surrogate,lr,val_macro_f1,val_macro_f1_std
0,arctan,0.0030,0.985788,0.012720
1,fast_sigmoid,0.0030,0.976968,0.011560
2,sigmoid,0.0001,0.979969,0.007390
3,triangular,0.0010,0.976708,0.021785



InsectSound: validation Macro-F1 by surrogate family and learning rate


lr,0.0001,0.0003,0.0010,0.0030
surrogate,,,,
arctan,0.0182,0.0182,0.0182,0.0196
fast_sigmoid,0.4160,0.4427,0.4175,0.4044
sigmoid,0.4098,0.4459,0.4590,0.4097
triangular,0.4497,0.3967,0.4405,0.4338


best learning rate per family:


,surrogate,lr,val_macro_f1,val_macro_f1_std
0,arctan,0.0030,0.019594,0.0
1,fast_sigmoid,0.0003,0.442724,0.0
2,sigmoid,0.0010,0.458983,0.0
3,triangular,0.0001,0.449724,0.0


## 3. Learning rate for the LSTM

The dense baseline has no neuron or surrogate parameters, so only its learning
rate is searched. Tuning it separately rather than imposing the spiking model's
rate is the fair choice: a single shared learning rate would penalise whichever
architecture happened to dislike it, and the resulting difference would be an
artefact of that choice rather than a property of the models.

In [7]:
lstm_config = {}

for name, (train_ds, _) in splits.items():
    cached = load_config(f"config_lstm_{name}")
    if cached is not None:
        lstm_config[name] = cached
        print(f"{name}: loaded cached configuration {cached}")
        continue

    table = grid_search(
        lambda **kw: build_model("lstm", num_classes=train_ds.num_classes,
                                 k=train_ds.seq_len, seed=SEED, **kw),
        train_ds, DEFAULT_GRIDS["lstm"], seed=SEED, desc=f"LSTM · {name}",
        train_kwargs={"batch_size": BATCH_SIZES[name]},
    )
    table.to_csv(f"{RESULTS_DIR}/search_lstm_{name}.csv", index=False)

    lstm_config[name] = {"lr": float(table.iloc[0].lr)}
    save_config(f"config_lstm_{name}", lstm_config[name])
    display(table)

LSTM · ECG5000:   0%|          | 0/10 [00:00<?, ?cfg/s]

  cfg 1/10 fold 1/5:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 1/10 fold 2/5:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 1/10 fold 3/5:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 1/10 fold 4/5:   0%|          | 0/200 [00:00<?, ?ep/s]

...

  cfg 7/10:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 8/10:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 9/10:   0%|          | 0/200 [00:00<?, ?ep/s]

  cfg 10/10:   0%|          | 0/200 [00:00<?, ?ep/s]

,lr,val_macro_f1,val_macro_f1_std,mean_best_epoch,stopped_early,n_splits
0,0.01000,0.275877,0.0,27,True,1
1,0.00200,0.230567,0.0,26,True,1
2,0.00005,0.125807,0.0,52,True,1
3,0.00020,0.123316,0.0,27,True,1
4,0.00010,0.122290,0.0,35,True,1
5,0.00050,0.084885,0.0,12,True,1
6,0.00100,0.084183,0.0,3,True,1
7,0.00002,0.067229,0.0,37,True,1
8,0.00001,0.063567,0.0,73,True,1
9,0.00500,0.056022,0.0,2,True,1


## 4. Untruncated reference performance

With the configurations fixed, each model is trained at $k = T$ under the
Gaussian initialization and evaluated on the test split. This is the only point
at which the test split is touched, and these figures are the reference against
which every truncated result in the following notebooks is read: they state what
each architecture achieves when it is given the entire sequence.

The stochastic conditions are repeated over five seeds so that later differences
can be judged against seed-to-seed variation.

In [8]:
configs = {"snn": snn_config, "lsm": substrate, "lstm": lstm_config}
reference = {}

for name, (train_ds, test_ds) in splits.items():
    conditions = [
        dict(kind=kind, k=train_ds.seq_len, init_method="gaussian", seed=seed,
             config=dict(configs[kind][name]))
        for kind in ("snn", "lsm", "lstm")
        for seed in range(5)
    ]
    reference[name] = run_grid(f"baselines_{name}", conditions, train_ds, test_ds)

baselines_ECG5000: 15 conditions to run


baselines_ECG5000:   0%|          | 0/15 [00:00<?, ?run/s]

baselines_Wafer: 15 conditions to run


baselines_Wafer:   0%|          | 0/15 [00:00<?, ?run/s]

baselines_InsectSound: 15 conditions to run


baselines_InsectSound:   0%|          | 0/15 [00:00<?, ?run/s]

In [9]:
summary = []
for name, frame in reference.items():
    grouped = frame.groupby("model").agg(
        macro_f1=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
        accuracy=("accuracy", "mean"), majority=("majority_rate", "mean"),
        macs=("macs", "mean"), sops=("sops", "mean"), energy_pj=("energy_pj", "mean"),
    ).reset_index()
    grouped.insert(0, "dataset", name)
    summary.append(grouped)

summary = pd.concat(summary, ignore_index=True)
summary["energy_uJ"] = summary.energy_pj / 1e6
display(summary.round(4))

,dataset,model,macro_f1,macro_f1_sd,accuracy,majority,macs,sops,energy_pj,energy_uJ
0,ECG5000,lsm,0.9496,0.0079,0.9512,0.5838,36352.0,328935.1000,4.632608e+05,0.4633
1,ECG5000,lstm,0.9624,0.0147,0.9640,0.5838,9246976.0,0.0000,4.253609e+07,42.5361
2,ECG5000,snn,0.9629,0.0078,0.9641,0.5838,36352.0,276658.3375,4.162117e+05,0.4162
3,Wafer,lsm,0.8929,0.0163,0.9600,0.8921,39424.0,369196.9312,5.136276e+05,0.5136
4,Wafer,lstm,0.7726,0.1729,0.9172,0.8921,10039552.0,0.0000,4.618194e+07,46.1819
5,Wafer,snn,0.9167,0.0352,0.9689,0.8921,39424.0,334717.8250,4.825964e+05,0.4826
6,InsectSound,lsm,0.3889,0.0064,0.4016,0.1000,156160.0,322433.6500,1.008526e+06,1.0085
7,InsectSound,lstm,0.3352,0.1731,0.3676,0.1000,39630080.0,0.0000,1.822984e+08,182.2984
8,InsectSound,snn,0.4174,0.0580,0.4417,0.1000,156160.0,267884.1469,9.594317e+05,0.9594


## 5. What this establishes

Three quantities carry forward. The selected configurations are frozen from this
point and are not revisited: every condition in the initialization and truncation
studies uses them unchanged, which is what allows those results to be attributed
to the initializer and the cutoff rather than to tuning.

The untruncated Macro-F1 of each model gives the ceiling against which its
truncated variants are compared, and the majority-class rate reported beside it
gives the floor. A model whose score sits near that floor has not learned the
minority class at all, which on Wafer is the only class of practical interest.

Finally, the operation counts at $k = T$ establish the untruncated energy of each
architecture. Since both the dense and spiking costs scale with the number of
time steps executed, these figures set the scale against which the savings from
truncation are measured in Notebook 05.